# AI-textdetektor
### Neuralt nätverk för att klassificera om en text är skriven av en människa eller AI

Den här notebooken täcker hela pipeline:
1. Importera bibliotek och konfiguration
2. Ladda data och sampla för EDA
3. Utforskande dataanalys (EDA)
4. Förbehandling — rensning, uppdelning, TF-IDF (ord + tecken), textstatistik, skalning
5. Bygga modellen
6. Träna modellen
7. Spara modellen
8. Utvärdera modellen
9. Feature importance (permutationsimportans)
10. Grid search (hyperparameteroptimering)
11. Testa modellen på en ny text
12. Starta webbappen (Streamlit)

---

---
## Bakgrund

Vi fick i uppdrag att utveckla ett verktyg som kan avgöra om en text är skriven av en människa eller genererad av en AI. Uppdraget kom inte från den håll man kanske förväntar sig.

Det vanliga sättet att tänka kring AI-textdetektering är att *avslöja* — att hitta fuskare, flagga automatgenererat innehåll eller skydda sig mot desinformation. Men våra uppdragsgivare hade ett annat perspektiv. Tänk dig en psykologmottagning som vill förbättra sin skriftliga kommunikation med patienter. Eller en vårdcentral i Norrland som använder AI för att formulera utskick, men märker att patienterna upplever texterna som opersonliga och svåra att relatera till. Frågan är inte längre *'är det här skrivet av en AI?'* — utan *'hur får vi det att låta mer mänskligt?'*

Det är den vändningen som gör det här projektet intressant. Verktyget fungerar som en spegel: det identifierar vad som skiljer mänskligt och AI-genererat skrivande åt, och den kunskapen kan användas för att göra AI-text varmare, mer personlig och lättare att ta till sig — särskilt i sammanhang där relationen mellan avsändare och mottagare spelar stor roll.

### Hur fungerar det?

Modellen analyserar text från tre håll samtidigt:

- **Ord-TF-IDF** fångar vilka ord och fraser som används. AI-text tenderar att återkomma till vissa formuleringar — *'det är viktigt att'*, *'sammanfattningsvis'*, *'det bör noteras'* — som sällan dyker upp i naturligt mänskligt skrivande.
- **Tecken-TF-IDF** tittar på mönster på teckennivå: interpunktion, radbrytningar, stavning. En människa som skriver snabbt gör andra val än en språkmodell som optimerar för koherens.
- **Textstatistik** mäter strukturella egenskaper — meningslängd, ordvariation, lexikal mångfald. AI-genererad text är ofta anmärkningsvärt jämn och välbalanserad, vilket paradoxalt nog avslöjar den.

De tre kanalerna kombineras och matas in i ett neuralt nätverk tränat på hundratusentals svenska och engelska texter. Resultatet är ett verktyg som inte bara säger *AI* eller *Människa* — utan som kan ge underlag för att förstå *varför* en text känns opersonlig, och vad som behöver förändras.

## 0. Importera bibliotek och konfiguration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from scikeras.wrappers import KerasClassifier

from config import *
from text_features import remove_source_leaks, compute_text_stats

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU tillgänglig: {tf.config.list_physical_devices('GPU')}")

---
## 1. Ladda data

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f"Datasetet har totalt {df.shape[0]:,} rader och {df.shape[1]} kolumner")
df.head()

In [ ]:
print("Labelfördelning i hela datasetet:")
print(df['label'].value_counts())
print(f"\nSaknade värden:\n{df.isnull().sum()}")

### 1.1 Sampla ett urval för EDA

Vi tar ett balanserat urval (styrt av `SAMPLES_PER_CLASS` i `config.py`) för att utforska datan. Träningen körs sedan på hela datasetet.

In [ ]:
df_human = df[df['label'] == 'Human'].sample(n=SAMPLES_PER_CLASS, random_state=RANDOM_SEED)
df_ai    = df[df['label'] == 'AI'].sample(n=SAMPLES_PER_CLASS, random_state=RANDOM_SEED)
df_sample = pd.concat([df_human, df_ai]).reset_index(drop=True)

print(f"EDA-urval: {len(df_sample):,} texter ({SAMPLES_PER_CLASS:,} per klass)")
df_sample['label'].value_counts()

---
## 2. Utforskande dataanalys (EDA)

In [ ]:
df_sample['text_length'] = df_sample['text'].str.len()
df_sample['word_count']  = df_sample['text'].str.split().str.len()

print("Textlängdsstatistik per klass:")
df_sample.groupby('label')[['text_length', 'word_count']].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Utforskande dataanalys', fontsize=16, fontweight='bold')

# Klassfördelning
df_sample['label'].value_counts().plot(
    kind='bar', ax=axes[0], color=['#008CFF', '#FF5722'])
axes[0].set_title('Klassfördelning', fontsize=14)
axes[0].set_xlabel('Klass')
axes[0].set_ylabel('Antal')
axes[0].tick_params(axis='x', rotation=0)

# Textlängd (tecken)
for label, color in zip(['Human', 'AI'], ['#2196F3', '#FF5722']):
    subset = df_sample[df_sample['label'] == label]
    axes[1].hist(subset['text_length'], bins=50, alpha=0.6, label=label, color=color)
axes[1].set_title('Fördelning av textlängd (tecken)', fontsize=14)
axes[1].set_xlabel('Antal tecken')
axes[1].set_ylabel('Frekvens')
axes[1].legend()

# Ordantal
for label, color in zip(['Human', 'AI'], ['#008CFF', '#FF5722']):
    subset = df_sample[df_sample['label'] == label]
    axes[2].hist(subset['word_count'], bins=50, alpha=0.6, label=label, color=color)
axes[2].set_title('Fördelning av ordantal', fontsize=14)
axes[2].set_xlabel('Antal ord')
axes[2].set_ylabel('Frekvens')
axes[2].legend()

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Sparad: eda_distributions.png")

---
## 3. Förbehandling

Träningen körs på **hela datasetet**. Viktigt: TF-IDF tränas **enbart på träningsdatan** för att undvika datainläckage.

### 3.1 Rensa data

`remove_source_leaks` tar bort publicistnamn (t.ex. "Reuters", "BBC") och URL:er som annars skulle ge modellen en genväg och förstöra generaliseringen.

In [ ]:
print(f"Före rensning: {df.shape}")
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].str.strip()
df['text'] = df['text'].apply(remove_source_leaks)
df = df[df['text'].str.len() > 0]
df = df.reset_index(drop=True)
print(f"Efter rensning: {df.shape}")

### 3.2 Dela upp i tränings- och testdata (på råtext)

Uppdelningen görs **före** TF-IDF-anpassningen, vilket förhindrar datainläckage från testdatan.

In [ ]:
X_text_train, X_text_test, y_train_raw, y_test_raw = train_test_split(
    df['text'], df['label'],
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=df['label']
)

print(f"Träningsdata: {X_text_train.shape[0]:,} texter")
print(f"Testdata:     {X_text_test.shape[0]:,} texter")

### 3.3 Koda om labels (anpassas enbart på träningsdata)

In [ ]:
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test  = le.transform(y_test_raw)

print("Label-kodning:")
for klass, kod in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {klass} → {kod}")

### 3.4 TF-IDF ord-kanal (anpassas enbart på träningsdata)

Fångar ord och bigramer — den klassiska text-representationen.

In [ ]:
print(f"Tränar ord-TF-IDF (max {TFIDF_MAX_FEATURES:,} features)...")
tfidf = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    stop_words=TFIDF_STOP_WORDS,
    ngram_range=TFIDF_NGRAM_RANGE,
    dtype=np.float32
)
X_train_word = tfidf.fit_transform(X_text_train)
X_test_word  = tfidf.transform(X_text_test)

print(f"Ord-TF-IDF-matris: {X_train_word.shape}")

### 3.5 TF-IDF tecken-kanal (anpassas enbart på träningsdata)

Fångar tecken-n-gram (3–4 tecken). Teckenmönster avslöjar stilistiska skillnader som ordnivån missar, t.ex. interpunktion och stavningsmönster.

In [ ]:
print(f"Tränar tecken-TF-IDF (max {TFIDF_CHAR_MAX_FEATURES:,} features)...")
tfidf_char = TfidfVectorizer(
    max_features=TFIDF_CHAR_MAX_FEATURES,
    analyzer='char_wb',
    ngram_range=TFIDF_CHAR_NGRAM_RANGE,
    dtype=np.float32
)
X_train_char = tfidf_char.fit_transform(X_text_train)
X_test_char  = tfidf_char.transform(X_text_test)

print(f"Tecken-TF-IDF-matris: {X_train_char.shape}")

### 3.6 Beräkna textstatistik

Tre strukturella mått per text: **ordantal**, **genomsnittlig meningslängd** och **lexikal mångfald**. Dessa fångar mönster som TF-IDF inte ser.

In [ ]:
print("Beräknar textstatistik...")
stats_train = compute_text_stats(X_text_train)
stats_test  = compute_text_stats(X_text_test)

print(f"Textstatistik-matris: {stats_train.shape}")
print(f"Kolumner: [word_count, avg_sent_len, lexical_diversity]")
print(f"Exempel (första tränings-texten): {stats_train[0]}")

### 3.7 Kombinera alla feature-kanaler

Sammanslagning med `hstack`: ord-TF-IDF + tecken-TF-IDF + textstatistik = en enda feature-matris.

In [ ]:
X_train = hstack([X_train_word, X_train_char, csr_matrix(stats_train)], format='csr')
X_test  = hstack([X_test_word,  X_test_char,  csr_matrix(stats_test)],  format='csr')

# TF:s SparseToDense kräver sorterade kolumnindex
X_train.sort_indices()
X_test.sort_indices()

print(f"Kombinerad feature-matris: {X_train.shape}")
print(f"  Ord-TF-IDF:      {TFIDF_MAX_FEATURES:>6,} features")
print(f"  Tecken-TF-IDF:   {TFIDF_CHAR_MAX_FEATURES:>6,} features")
print(f"  Textstatistik:   {3:>6,} features")

# Frigör enskilda kanalmatriser
del X_train_word, X_train_char, stats_train
del X_test_word,  X_test_char,  stats_test

### 3.8 Skala features (anpassas enbart på träningsdata)

In [ ]:
scaler  = StandardScaler(with_mean=False, copy=False)
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
print("Skalning klar.")

---
## 4. Bygga modellen

Arkitekturen styrs av `HIDDEN_LAYERS` i `config.py`. Varje lager har L2-regularisering för att motverka överanpassning.

In [ ]:
model = tf.keras.models.Sequential()
model.add(tf.keras.Input(shape=(X_train.shape[1],)))

for units, activation, dropout_rate in HIDDEN_LAYERS:
    model.add(
        tf.keras.layers.Dense(
            units,
            activation=activation,
            kernel_regularizer=tf.keras.regularizers.L2(1e-4)
        )
    )
    if dropout_rate > 0:
        model.add(tf.keras.layers.Dropout(dropout_rate))

model.add(tf.keras.layers.Dense(1, activation=OUTPUT_ACTIVATION))

if LEARNING_RATE is not None:
    optimizer = tf.keras.optimizers.get({
        'class_name': OPTIMIZER,
        'config': {'learning_rate': LEARNING_RATE}
    })
else:
    optimizer = OPTIMIZER

model.compile(optimizer=optimizer, loss=LOSS, metrics=['accuracy'])
model.summary()

---
## 5. Träna modellen

Valideringsdatan skapas med en explicit `train_test_split` istället för `validation_split=` — detta ger stratifierad uppdelning och mer kontroll.

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor=ES_MONITOR,
    mode=ES_MODE,
    patience=ES_PATIENCE,
    restore_best_weights=ES_RESTORE_BEST,
    verbose=1
)

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train,
    test_size=VALIDATION_SPLIT,
    random_state=RANDOM_SEED,
    stratify=y_train
)

history = model.fit(
    X_fit, y_fit,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=1
)

---
## 6. Spara modellen

In [ ]:
model.save('ann_ai_detector_model.keras')
joblib.dump(tfidf,      'tfidf_vectorizer.joblib')
joblib.dump(tfidf_char, 'tfidf_char_vectorizer.joblib')
joblib.dump(le,         'label_encoder.joblib')
joblib.dump(scaler,     'scaler.joblib')

print("Sparade filer:")
for f in ['ann_ai_detector_model.keras', 'tfidf_vectorizer.joblib',
          'tfidf_char_vectorizer.joblib', 'label_encoder.joblib', 'scaler.joblib']:
    print(f"  {f}")

---
## 7. Utvärdera modellen

In [ ]:
y_pred_prob = model.predict(X_test).flatten()
y_pred      = (y_pred_prob >= 0.5).astype(int)

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Testförlust:     {test_loss:.4f}")
print(f"Testnoggrannhet: {test_accuracy:.4f}")

print("\nKlassificeringsrapport:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

### 7.1 Träningskurvor

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Träningshistorik', fontsize=16, fontweight='bold')

axes[0].plot(history.history['loss'],     label='Träningsförlust',    color='#2196F3')
axes[0].plot(history.history['val_loss'], label='Valideringsförlust', color='#FF5722')
axes[0].set_title('Förlust per epok', fontsize=14)
axes[0].set_xlabel('Epok')
axes[0].set_ylabel('Förlust')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'],     label='Träningsnoggrannhet',    color='#2196F3')
axes[1].plot(history.history['val_accuracy'], label='Valideringsnoggrannhet', color='#FF5722')
axes[1].set_title('Noggrannhet per epok', fontsize=14)
axes[1].set_xlabel('Epok')
axes[1].set_ylabel('Noggrannhet')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Sparad: training_curves.png")

### 7.2 Konfusionsmatris

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title('Konfusionsmatris', fontsize=14)
ax.set_xlabel('Förutsagd klass')
ax.set_ylabel('Verklig klass')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Sparad: confusion_matrix.png")

### 7.3 ROC-kurva och Precision-Recall-kurva

In [ ]:
fpr, tpr, _          = roc_curve(y_test, y_pred_prob)
roc_auc              = auc(fpr, tpr)
precision, recall, _ = precision_recall_curve(y_test, y_pred_prob)
avg_precision        = average_precision_score(y_test, y_pred_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Modellprestanda', fontsize=16, fontweight='bold')

axes[0].plot(fpr, tpr, color='#2196F3', lw=2, label=f'ROC-kurva (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
axes[0].set_title('ROC-kurva', fontsize=14)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

axes[1].plot(recall, precision, color='#FF5722', lw=2, label=f'PR-kurva (AP = {avg_precision:.4f})')
axes[1].set_title('Precision-Recall-kurva', fontsize=14)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Sparad: roc_pr_curves.png")

---
## 8. Feature importance (permutationsimportans)

Vilka features driver modellens beslut? Vi kombinerar namn från alla tre kanalerna (ord-TF-IDF, tecken-TF-IDF, textstatistik) och kör en tvåstegs-permutationsanalys:

- **Steg 1:** Förhandsgallra kandidater via modellvikterna (snabbt).
- **Steg 2:** Blanda en feature i taget och mät noggrannhetsminskningen (mer exakt).

In [ ]:
combined_feature_names = (
    list(tfidf.get_feature_names_out()) +
    list(tfidf_char.get_feature_names_out()) +
    ['word_count', 'avg_sent_len', 'lexical_diversity']
)
print(f"Totalt antal features: {len(combined_feature_names):,}")

In [ ]:
# Steg 1: Förhandsgallra via modellvikter
first_layer_weights = np.abs(model.layers[0].get_weights()[0])
weight_importance   = first_layer_weights.sum(axis=1)
candidate_idx       = weight_importance.argsort()[-FI_CANDIDATES:]
feature_names_arr   = np.asarray(combined_feature_names)
print(f"Förhandsgallrade {FI_CANDIDATES} kandidatfeatures från modellvikter")

# Steg 2: Permutationsimportans på kandidaterna
X_test_small = X_test[:FI_TEST_SAMPLES]
y_test_small = y_test[:FI_TEST_SAMPLES]

y_base_prob  = model.predict(X_test_small, verbose=0).flatten()
baseline_acc = np.mean((y_base_prob >= 0.5).astype(int) == y_test_small)
print(f"Basnoggrannhet på urval: {baseline_acc:.4f}")

importances = np.zeros(FI_CANDIDATES)
for idx, feat_i in enumerate(tqdm(candidate_idx, desc="Beräknar permutationsimportans")):
    drops = []
    for r in range(FI_REPEATS):
        X_permuted = X_test_small.copy()
        np.random.seed(r)
        col = np.asarray(X_permuted[:, feat_i].todense()).flatten()
        X_permuted[:, feat_i] = np.random.permutation(col).reshape(-1, 1)
        y_perm_prob = model.predict(X_permuted, verbose=0).flatten()
        perm_acc    = np.mean((y_perm_prob >= 0.5).astype(int) == y_test_small)
        drops.append(baseline_acc - perm_acc)
    importances[idx] = np.mean(drops)

top_local_idx   = importances.argsort()[-FI_TOP_N:]
top_feature_idx = candidate_idx[top_local_idx]
top_importances = importances[top_local_idx]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(FI_TOP_N), top_importances, color='#2196F3')
ax.set_yticks(range(FI_TOP_N))
ax.set_yticklabels(feature_names_arr[top_feature_idx])
ax.set_title(f'Topp {FI_TOP_N} viktigaste features (permutationsimportans)', fontsize=14)
ax.set_xlabel('Genomsnittlig minskning i noggrannhet')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Sparad: feature_importance.png")

---
## 9. Grid search (hyperparameteroptimering)

Grid search körs på **EDA-urvalet** (`df_sample`) för att hålla körtiden rimlig. Målet är att hitta de bästa hyperparametrarna — träna sedan den slutliga modellen en gång på hela datasetet med de vinnande inställningarna.

Parameternätet definieras av `PARAM_GRID` i `config.py`.

> **Obs:** Kör cellen nedan separat — den är oberoende av träningsstegen ovan.

In [ ]:
# ── Förbered data för grid search (EDA-urval) ──────────────────────────────
df_gs = df_sample.copy()
df_gs = df_gs.dropna(subset=['text', 'label'])
df_gs['text'] = df_gs['text'].str.strip().apply(remove_source_leaks)
df_gs = df_gs[df_gs['text'].str.len() > 0].reset_index(drop=True)

X_gs_text_train, X_gs_text_test, y_gs_train_raw, y_gs_test_raw = train_test_split(
    df_gs['text'], df_gs['label'],
    test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=df_gs['label']
)

le_gs = LabelEncoder()
y_gs_train = le_gs.fit_transform(y_gs_train_raw)

tfidf_gs = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, stop_words=TFIDF_STOP_WORDS,
                            ngram_range=TFIDF_NGRAM_RANGE, dtype=np.float32)
tfidf_char_gs = TfidfVectorizer(max_features=TFIDF_CHAR_MAX_FEATURES, analyzer='char_wb',
                                 ngram_range=TFIDF_CHAR_NGRAM_RANGE, dtype=np.float32)

X_gs_word  = tfidf_gs.fit_transform(X_gs_text_train)
X_gs_char  = tfidf_char_gs.fit_transform(X_gs_text_train)
X_gs_stats = compute_text_stats(X_gs_text_train)

X_gs = hstack([X_gs_word, X_gs_char, csr_matrix(X_gs_stats)], format='csr')
X_gs.sort_indices()

scaler_gs = StandardScaler(with_mean=False, copy=False)
X_gs = scaler_gs.fit_transform(X_gs)

print(f"Grid search-matris: {X_gs.shape}")

In [ ]:
# ── Bygg modellen för grid search ──────────────────────────────────────────
def build_model(n_neurons_layer1=32, n_neurons_layer2=16,
                dropout_rate=0.3, activation='relu',
                learning_rate=LEARNING_RATE, optimizer=OPTIMIZER):
    m = tf.keras.models.Sequential()
    m.add(tf.keras.Input(shape=(X_gs.shape[1],)))
    m.add(tf.keras.layers.Dense(n_neurons_layer1, activation=activation))
    m.add(tf.keras.layers.Dropout(dropout_rate))
    m.add(tf.keras.layers.Dense(n_neurons_layer2, activation=activation))
    m.add(tf.keras.layers.Dropout(dropout_rate))
    m.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    opt = tf.keras.optimizers.get({
        'class_name': optimizer,
        'config': {'learning_rate': learning_rate}
    })
    m.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    return m

estimator = KerasClassifier(model=build_model, verbose=0, random_state=RANDOM_SEED)

grid = GridSearchCV(
    estimator=estimator,
    param_grid=PARAM_GRID,
    n_jobs=2,
    cv=3,
    scoring='balanced_accuracy',
    verbose=1
)

grid_result = grid.fit(X_gs, y_gs_train)

print("\nBästa parametrar:", grid_result.best_params_)
print(f"Bästa resultat (balanced accuracy): {grid_result.best_score_:.4f}")

In [ ]:
# ── Sammanfattning av alla kombinationer ───────────────────────────────────
gs_results = pd.DataFrame(grid_result.cv_results_)
gs_results = gs_results[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']]
gs_results = gs_results.sort_values('rank_test_score')
print("Topp 10 kombinationer:")
gs_results.head(10)

---
## 10. Testa modellen på en ny text

Skriv in valfri text och se om modellen klassificerar den som AI eller människa. Cellen använder de sparade artefakterna.

In [ ]:
test_text = "Skriv din text här och se vad modellen säger."

# Rensa och vektorisera
text_clean  = remove_source_leaks(test_text.strip())
X_w         = tfidf.transform([text_clean])
X_c         = tfidf_char.transform([text_clean])
X_s         = compute_text_stats([text_clean])
X_input     = hstack([X_w, X_c, csr_matrix(X_s)], format='csr')
X_input.sort_indices()
X_input     = scaler.transform(X_input)

# Förutsäg
prob  = model.predict(X_input, verbose=0).flatten()[0]
klass = le.inverse_transform([(prob >= 0.5).astype(int)])[0]

print(f"Text:          {test_text[:100]}" + ("..." if len(test_text) > 100 else ""))
print(f"Förutsägelse:  {klass}")
print(f"Sannolikhet:   {prob:.4f}  (≥0.5 → AI, <0.5 → Människa)")

# Textstatistik
stats = compute_text_stats([text_clean])[0]
print(f"\nTextstatistik:")
print(f"  Ordantal:          {int(stats[0])}")
print(f"  Genomsn. meningsl: {stats[1]:.1f} ord")
print(f"  Lexikal mångfald:  {stats[2]:.1f}%")

---
## 11. Starta webbappen (Streamlit)

`app.py` är en interaktiv webbapp byggd med Streamlit. Den laddar den tränade modellen och låter användaren klistra in valfri text och få ett svar direkt i webbläsaren — utan att behöva köra någon kod.

**Förutsättningar:** Modellen måste vara tränad och sparad (kör avsnitten 1–6 ovan, eller använd en redan sparad modell).

### Starta appen

Öppna en terminal i projektmappen och kör:

```bash
streamlit run app.py
```

Appen öppnas automatiskt i webbläsaren på `http://localhost:8501`.

### Vad appen gör

1. Laddar modellen och alla artefakter (`tfidf_vectorizer.joblib`, `tfidf_char_vectorizer.joblib`, `label_encoder.joblib`, `scaler.joblib`).
2. Tar emot valfri text via ett textfält.
3. Kör samma förbehandlingspipeline som träningen: källrensning → ord-TF-IDF → tecken-TF-IDF → textstatistik → skalning.
4. Visar ett resultat (*Likely AI* / *Likely Human* / *Inconclusive*) med konfidenspoäng och de ord som påverkade beslutet mest.
5. Erbjuder nedladdning av en analysrapport som CSV.

> **Demo-läge:** Om modellfilerna saknas kan appen köras i demo-läge med slumpmässiga förutsägelser — aktiveras automatiskt eller via kryssrutan i sidofältet.